In [28]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
#from transformers import BitsAndBytesConfig
import spacy
import torch
from yahooquery import search
from spacy.language import Language
from spacy import displacy
import time
import glob
import re
import math
import statistics
import os
import json
import calendar
import holidays
from pathlib import Path
from datetime import date
from datetime import datetime
import pandas as pd
import numpy as np
import collections
import hashlib
from dateutil.parser import parse
import shutil
import ast
from io import StringIO
import requests
import glob
import os

In [30]:
alias_file = "../../Summary/OTHER/aliases.json"
if os.path.exists(alias_file):
    with open(alias_file, 'r') as f:
        alias = json.load(f)
print(alias)

{'MAUS': 'MONTHLY ACTIVE USERS', 'ARR': 'ANNUAL RECURRING REVENUE', 'ARPU': 'ACTIVE REVENUE PER USER', 'ANNUAL RECURRING REVENUE ARR': 'ANNUAL RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE': 'ANNUAL RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE ARR': 'ANNUAL RECURRING REVENUE', 'NET NEW ARR': 'NET NEW ANNUAL RECURRING REVENUE', 'NET DOLLAR EXPANSION CUSTOMERS WITH MORE THAN 10 EMPLOYEES': 'NET DOLLAR EXPANSION', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 TTM REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS WITH MORE THAN $100000 OF ARR': 'LARGE PAID CUSTOMERS', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 IN REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS WITH MORE THAN 10 EMPLOYEES': 'TOTAL NUMBER OF PAID CUSTOMERS', 'BUSINESSES': 'TOTAL NUMBER OF PAID CUSTOMERS', 'NET NEW SUBSCRIPTION CUSTOMERS': 'NEW PAID CUSTOMERS', 'SUBSCRIPTION CUSTOMERS': 'TOTAL NUMBER OF PAID CUSTOMERS', 'CUSTOMER COUNT': 'TOTAL NUMBER 

In [31]:
reverse_alias = dict()
for key in alias.keys():
    val = alias[key]
    if(val not in reverse_alias):
        reverse_alias[val] = list()
    reverse_alias[val].append(key)
print(reverse_alias)

{'MONTHLY ACTIVE USERS': ['MAUS', 'MAUS-GLOBAL', 'GLOBAL MONTHLY ACTIVE USERS MAUS', 'GLOBAL MONTHLY ACTIVE USERS'], 'ANNUAL RECURRING REVENUE': ['ARR', 'ANNUAL RECURRING REVENUE ARR', 'ANNUALIZED RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE ARR', 'ANNUAL RUN-RATE REVENUE ARR', 'ANNUALIZED EXIT MONTHLY RECURRING SUBSCRIPTIONS ARR', 'RINGCENTRAL TOTAL ARR', 'ENDING ARR'], 'ACTIVE REVENUE PER USER': ['ARPU', 'AVERAGE REVENUE PER CUSTOMER ARPU', 'AVERAGE REVENUE PER CUSTOMER', 'ARPU-GLOBAL'], 'NET NEW ANNUAL RECURRING REVENUE': ['NET NEW ARR'], 'NET DOLLAR EXPANSION': ['NET DOLLAR EXPANSION CUSTOMERS WITH MORE THAN 10 EMPLOYEES', 'NET DOLLAR EXPANSION CUSTOMERS WITH GREATER THAN 10 EMPLOYEES', 'REVENUE RETENTION', 'NET DOLLAR EXPANSION TOTAL NUMBER OF CUSTOMERS', 'RETENTION RATE', 'SUBSCRIPTION REVENUE RETENTION RATE', 'NET REVENUE RETENTION', 'NET DOLLAR - BASED RETENTION RATE', 'SUBSCRIPTION REVENUE NET DOLLAR EXPANSION', 'NET DOLLAR RETENTION NDR', 'NET DOLLAR RETENTION', 'NET RET

In [32]:
org_to_sym = dict()

In [33]:
maxEntCount = 12
naStr = "Not enough information is available."
ansTemplateStr = dict()
ansTemplateStr["1"] = "*METRIC of *ORG in quarter *QTR year *YEAR was *RESULT. " 
ansTemplateStr["2"] = "*ORG reported *COUNT *METRIC in quarter *QTR year *YEAR, these were:\n*RESULT"
ansTemplateStr["3"] = "*RESULT. *ORG *CHG *METRIC consensus guidance in *QTR *YEAR. *METRIC is *NUMBER, *CHG by *MARGIN in *QTR *YEAR."
ansTemplateStr["4"] = "*RESULT. *ORG *CHG *METRIC consensus guidance for *GQTR *GYR. *METRIC guidance is *NUMBER, *CHG consensus guidance by *MARGIN in *GQTR *GYR."
ansTemplateStr["5"] = "*RESULT. *ORG *CHG *METRIC guidance for full fiscal year *GYR. *METRIC midpoint guidance is *NUMBER, *CHG by *MARGIN for fiscal year *GYR."


In [34]:
def getOrgData(org):
    orgDataPath = "../../Summary/orgData/"+org+".txt"
    file = Path(orgDataPath)
    if file.is_file():
        #print(True)
        with open(orgDataPath) as f:
            data = json.load(f)
        #print(data)
        return data
    return None

In [35]:
def getOrgAttr(orgData, attr):
    if not orgData:
        return None
    asplit = attr.split("|")
    parent = asplit[0]
    if parent in orgData and "SOURCE" in orgData[parent]:
        src = orgData[parent]["SOURCE"]
        if src == "YH" or (parent == "ORGPROFILE" and src == "AD"):
            p = orgData
            for i in range(0, len(asplit)):
                if asplit[i] not in p:
                    return None
                p = p[asplit[i]]
            #print(p)
            return(p)
    return None

In [36]:
def getPrevQtr(qstr):
    if not qstr:
        return None
    prvQtr = None
    qs = qstr.split("-")[0]
    year = qstr.split("-")[1]
    if(qs == "Q1"):
        year = (int(year) - 1)
        prvQtr = "Q4-"+str(year)
    elif(qs == "Q2"):
        prvQtr = "Q1-"+str(year)
    elif(qs == "Q3"):
        prvQtr = "Q2-"+str(year)
    elif(qs == "Q4"):
        prvQtr = "Q3-"+str(year)
    return(prvQtr)

In [37]:
def getEntAttr(entData, attr):
    if not entData:
        return None
    asplit = attr.split("|")
    
    p = entData
    for i in range(0, len(asplit)):
        if asplit[i] not in p:
            return None
        p = p[asplit[i]]
    #print(p)
    return(p)

In [38]:
def getAttr(allEntities, source, attrList):
    if source not in allEntities:
        return None
    p = allEntities[source]
    if not attrList:
        return p
    asplit = attrList.split("|")
    for i in range(0, len(asplit)):
        if asplit[i] not in p:
            return None
        p = p[asplit[i]]
        #print(p)
    return(p)
    #return None

In [39]:
def isMetricPresent(source, metric):
    if metric not in source:
        return False
    if metric in source and "CONFLICT" in source[metric] and source[metric]["CONFLICT"]:
        return False
    return True

In [40]:
def name_to_symbol(company_name):
    results = search(company_name)
    if(results and "quotes" in results):
        return results["quotes"][0]["symbol"]
    return None

In [41]:
def get_earning_details(csym):
    entPath = "../../Summary/entities/"+csym+"-ENTITIES.json"
    entFile = Path(entPath)
    entities = None
    with open(entPath, encoding="utf-8") as f:
        entity = json.load(f)
        entities = entity[csym]
    return(entities)

In [42]:
def getEntities(sym):
    entPath = "../../Summary/entities/"+sym+"-ENTITIES.json"
    entFile = Path(entPath)
    entities = None
    allEntities = None
    if entFile.is_file():
        with open(entPath, encoding="utf-8") as f:
            entity = json.load(f)
            entities = entity[sym]
            allEntities = dict()
            orgData = getOrgData(sym)
            allEntities["ORGDATA"] = orgData
            allEntities["ENTITIES"] = entities
            #allEntities["ENTITY"] = entity
    return allEntities

In [43]:
import ast
import operator

# Supported operators
OPS = {
    ast.Eq: operator.eq,
    ast.NotEq: operator.ne,
    ast.Lt: operator.lt,
    ast.LtE: operator.le,
    ast.Gt: operator.gt,
    ast.GtE: operator.ge,
    ast.And: lambda a, b: a and b,
    ast.Or: lambda a, b: a or b,
}

def normalize_keys(obj):
    if isinstance(obj, dict):
        return {k.replace("-", "_").replace(" ","_"): normalize_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [normalize_keys(i) for i in obj]
    return obj

def deep_get(data, dotted_path):
    """Retrieve nested key using dot notation safely."""
    keys = dotted_path.split(".")
    for k in keys:
        if isinstance(data, dict):
            data = data.get(k)
        elif isinstance(data, list):
            try:
                data = data[int(k)]
            except (ValueError, IndexError):
                return None
        else:
            return None
    return data

def safe_eval_expr(node, context):
    """Recursively evaluate AST expression safely using nested JSON context."""
    try:
        if isinstance(node, ast.BoolOp):  # logical and/or
            left = safe_eval_expr(node.values[0], context)
            for v in node.values[1:]:
                right = safe_eval_expr(v, context)
                left = OPS[type(node.op)](left, right)
            return left

        elif isinstance(node, ast.Compare):  # comparisons
            left = safe_eval_expr(node.left, context)
            right = safe_eval_expr(node.comparators[0], context)
            op = OPS.get(type(node.ops[0]))
            if op is None:
                raise ValueError(f"Unsupported operator: {ast.dump(node.ops[0])}")
            return op(left, right)

        elif isinstance(node, ast.Attribute):
            # Handle dotted attribute like metrics.revenue.amount
            return deep_get(context, get_full_attr_path(node))

        elif isinstance(node, ast.Name):
            # top-level variable
            return context.get(node.id, None)

        elif isinstance(node, ast.Constant):
            return node.value

        elif hasattr(ast, "Num") and isinstance(node, ast.Num):
            return node.n

        elif hasattr(ast, "Str") and isinstance(node, ast.Str):
            return node.s

        else:
            return None
            #raise ValueError(f"Unsupported expression: {ast.dump(node)}")
    except:
        return None

def get_full_attr_path(node):
    """Reconstruct dotted attribute path."""
    parts = []
    while isinstance(node, ast.Attribute):
        parts.append(node.attr)
        node = node.value
    if isinstance(node, ast.Name):
        parts.append(node.id)
    return ".".join(reversed(parts))

def apply_json_filter(data, condition_str):
    """Apply logical filter to nested JSON data."""
    try:
        expr = ast.parse(condition_str, mode="eval").body
        return safe_eval_expr(expr, data)
    except:
        return False

In [44]:
entities = getEntities("APPN")
latest_qtr = entities["ENTITIES"]["LATEST-QTR"]
allData = entities["ENTITIES"][latest_qtr]
allData = normalize_keys(allData)
filter_expr = 'ANY.TEXT_METRICTYPE=="OPMETRIC"'
for ft in allData:
    nfilter_expr = filter_expr.replace("ANY.",ft+".")
    #print(nfilter_expr)
    result = apply_json_filter(allData, nfilter_expr)
    #print(nfilter_expr, result)
    if(result == True):
        print(ft)

SUBSCRIPTION_REVENUE_CLOUD
SUBSCRIPTIONS_REVENUE
PROFESSIONAL_SERVICES_REVENUE
NET_DOLLAR_EXPANSION


In [45]:
def list_quarter_range(start_qtr, end_qtr, calendar="QUARTERLY"):
    """
    Generate list of quarters in Qx-YYYY format between given start and end (inclusive).
    Example: Q2-2023 → Q4-2024
    """
    def parse_qtr(qtr_str):
        q, y = qtr_str.split('-')
        if(q == "ALL" and calendar == "QUARTERLY"):
            q = "Q1"
        if(calendar == "QUARTERLY"):
            return int(q[1]), int(y)  # (quarter_num, year)
        else:
            return "ALL", int(y) 

    start_q, start_y = parse_qtr(start_qtr)
    end_q, end_y = parse_qtr(end_qtr)
    #print(start_q, start_y, end_q, end_y)
    
    if((start_q == end_q == 1) and "ALL-" in start_qtr and "ALL-" in end_qtr):
        end_q = 4
    elif("ALL-" not in start_qtr and "ALL-" in end_qtr):
        end_q = 4

    result = []
    y, q = start_y, start_q

    if(calendar == "QUARTERLY"):
        while (y < end_y) or (y == end_y and q <= end_q):
            result.append(f"Q{q}-{y}")
            q += 1
            if q > 4:
                q = 1
                y += 1
    else:
        while (y < end_y or (y == end_y)):
            result.append(f"{q}-{y}")
            y += 1

    return result

In [46]:
print(list_quarter_range("Q2-2023", "Q2-2025", "QUARTERLY"))

['Q2-2023', 'Q3-2023', 'Q4-2023', 'Q1-2024', 'Q2-2024', 'Q3-2024', 'Q4-2024', 'Q1-2025', 'Q2-2025']


In [113]:
key_alias = dict()
key_alias["POSFACTS"] = "POSITIVE FACTS"
key_alias["NEGFACTS"] = "NEGATIVE FACTS"
#key_alias["POSITIVE FACTS"] = "POSFACTS"
#key_alias["NEGATIVE FACTS"] = "NEGFACTS"
#key_alias["POSITIVE POINTS"] = "POSFACTS"
#key_alias["NEGATIVE POINTS"] = "NEGFACTS"
#key_alias["EPS YOY"] = "EPS-YOY"
print(key_alias)

{'POSFACTS': 'POSITIVE FACTS', 'NEGFACTS': 'NEGATIVE FACTS'}


In [125]:
key_to_fields_alias = dict()
key_to_fields_alias["POSITIVE FACTS"] = ['POSFACTS']
key_to_fields_alias["NEGATIVE FACTS"] = ['NEGFACTS']
key_to_fields_alias["BEAT *KEY EXPECTATION?"] = ['*KEY-*-RESULT', '*KEY-*']
key_to_fields_alias["BEAT *KEY GUIDANCE?"] = ['*KEY-*-RESULT', '*KEY-*']
key_to_fields_alias["RAISE *KEY GUIDANCE?"] = ['*KEY-*-RESULT', '*KEY-*']
key_to_fields_alias["OPERATIONAL METRICS"] = ["OPERATIONAL METRICS"]
key_to_fields_alias["*KEY GROWTH"] = ['*KEY', '*KEY-QOQ', '*KEY-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']
key_to_fields_alias["*KEY GROWTH QUARTERLY"] = ['*KEY', '*KEY-QOQ', '*KEY-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']
key_to_fields_alias["*KEY GROWTH YEARLY"] = ['*KEY', '*KEY-YOY', 'GUIDANCE', 'GUIDE-YOY']
key_to_fields_alias["*KEY"] = ['*KEY', '*KEY-QOQ', '*KEY-YOY']
key_to_fields_alias["YOY *KEY GROWTH"] = ['*KEY', '*KEY-QOQ', '*KEY-YOY']
key_to_fields_alias["QOQ *KEY GROWTH"] = ['*KEY', '*KEY-QOQ', '*KEY-YOY']
key_to_fields_alias["YEARLY *KEY VS YEARLY GUIDANCE"] = ["*KEY-FULLYEAR","*KEY-FULLYEAR-YOY","*KEY-GUIDEFULL","*KEY-GUIDEFULL-YOY"]
"""
key_to_fields_alias["BEAT REVENUE EXPECTATION?"] = ['REVENUE-*-RESULT', 'REVENUE-*']
key_to_fields_alias["BEAT EPS EXPECTATION?"] = ['EPS-*-RESULT', 'EPS-*']
key_to_fields_alias["BEAT EPS GUIDANCE?"] = ['EPS-*-RESULT', 'EPS-*']
key_to_fields_alias["BEAT REVENUE GUIDANCE?"] = ['REVENUE-*-RESULT', 'REVENUE-*']
key_to_fields_alias["RAISE REVENUE GUIDANCE?"] = ['REVENUE-*-RESULT', 'REVENUE-*']
key_to_fields_alias["RAISE EPS GUIDANCE?"] = ['EPS-*-RESULT', 'EPS-*']
key_to_fields_alias["OPERATIONAL METRICS"] = ["OPERATIONAL METRICS"]
key_to_fields_alias["REVENUE GROWTH"] = ['REVENUE', 'REVENUE-QOQ', 'REVENUE-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']
key_to_fields_alias["REVENUE GROWTH QUARTERLY"] = ['REVENUE', 'REVENUE-QOQ', 'REVENUE-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']
key_to_fields_alias["REVENUE GROWTH YEARLY"] = ['REVENUE', 'REVENUE-YOY', 'GUIDANCE', 'GUIDE-YOY']
key_to_fields_alias["GAAP-EPS GROWTH"] = ['GAAP-EPS', 'GAAP-EPS-QOQ', 'GAAP-EPS-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']
key_to_fields_alias["GAAP-EPS GROWTH QUARTERLY"] = ['GAAP-EPS', 'GAAP-EPS-QOQ', 'GAAP-EPS-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']
key_to_fields_alias["GAAP-EPS GROWTH YEARLY"] = ['GAAP-EPS', 'GAAP-EPS-YOY', 'GUIDANCE', 'GUIDE-YOY']
key_to_fields_alias["REVENUE"] = ['REVENUE', 'REVENUE-QOQ', 'REVENUE-YOY']
key_to_fields_alias["YOY EPS GROWTH"] = ['EPS', 'EPS-QOQ', 'EPS-YOY']
"""
print(key_to_fields_alias)

{'POSITIVE FACTS': ['POSFACTS'], 'NEGATIVE FACTS': ['NEGFACTS'], 'BEAT *KEY EXPECTATION?': ['*KEY-*-RESULT', '*KEY-*'], 'BEAT *KEY GUIDANCE?': ['*KEY-*-RESULT', '*KEY-*'], 'RAISE *KEY GUIDANCE?': ['*KEY-*-RESULT', '*KEY-*'], 'OPERATIONAL METRICS': ['OPERATIONAL METRICS'], '*KEY GROWTH': ['*KEY', '*KEY-QOQ', '*KEY-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY'], '*KEY GROWTH QUARTERLY': ['*KEY', '*KEY-QOQ', '*KEY-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY'], '*KEY GROWTH YEARLY': ['*KEY', '*KEY-YOY', 'GUIDANCE', 'GUIDE-YOY'], '*KEY': ['*KEY', '*KEY-QOQ', '*KEY-YOY'], 'YOY *KEY GROWTH': ['*KEY', '*KEY-QOQ', '*KEY-YOY'], 'QOQ *KEY GROWTH': ['*KEY', '*KEY-QOQ', '*KEY-YOY'], 'YEARLY *KEY VS YEARLY GUIDANCE': ['*KEY-FULLYEAR', '*KEY-FULLYEAR-YOY', '*KEY-GUIDEFULL', '*KEY-GUIDEFULL-YOY']}


In [126]:
def buildArgsFromGenData(genData):
    arg_dict = dict()
    new_arg_dict = dict()
    
    arglist = genData["ARGS"].split("!!")
    for arg in arglist:
        argkey = arg.split(":")[0]
        argval = arg.split(":")[1]
        if(argkey not in arg_dict):
            arg_dict[argkey] = argval
    print(arg_dict)
    
    orgs = None
    if("ORG" in arg_dict and arg_dict["ORG"] != "NA"):
        orgs = arg_dict["ORG"]
        org_list = orgs.split("&&")
    else:
        return None # Add memory in case of chatbot implementation
    
    new_arg_dict["ORGS"] = list()
        
    for org in org_list:
        new_arg_dict[org] = dict()
        new_arg_dict["ORGS"].append(org)
        new_arg_dict[org]["SYM"] = arg_dict["ORG"]
        if(org not in org_to_sym):
            sym = name_to_symbol(org)
            if(sym):
                org_to_sym[org] = sym
                new_arg_dict[org]["SYM"] = sym
        else:
            new_arg_dict[org]["SYM"] = org_to_sym[org]
        sym = new_arg_dict[org]["SYM"]
      
        keys = None
        if("KEY" in arg_dict and arg_dict["KEY"] != "NA"):
            new_arg_dict["RALIAS"] = dict()
            keys = arg_dict["KEY"].split("&&")
            new_arg_dict[org]["KEY"] = list()
            for key in keys:
                if(key != "GUIDE-YOY"):
                    nkey = key.replace("-YOY","").replace("-QOQ","")
                else:
                    nkey = key
                if(nkey in key_alias):
                    rkey = key_alias[nkey]
                    new_arg_dict[org]["KEY"].append(rkey)
                else:
                    rkey = nkey
                    new_arg_dict[org]["KEY"].append(rkey)
                new_arg_dict["RALIAS"][rkey] = nkey
        else:
            return None # Add memory in case of chatbot implementation

        if("CALENDAR" in arg_dict and arg_dict["CALENDAR"] != "NA"):
            if(arg_dict["CALENDAR"] == "Q"):
                new_arg_dict[org]["CALENDAR"] = "QUARTERLY"
            elif(arg_dict["CALENDAR"] == "Y"):
                new_arg_dict[org]["CALENDAR"] = "YEARLY"
        else:
            return None # Add memory in case of chatbot implementation
        
        if("FROM" not in arg_dict or ("FROM" in arg_dict and arg_dict["FROM"] == "NA")):
            return None # Add memory in case of chatbot implementation
        
        new_arg_dict[org]["FROM"] = arg_dict["FROM"]
        
        if("TO" not in arg_dict or ("TO" in arg_dict and arg_dict["TO"] == "NA")):
            return None # Add memory in case of chatbot implementation
        
        new_arg_dict[org]["TO"] = arg_dict["TO"]

        if("SECTION" in arg_dict):
            new_arg_dict[org]["SECTION"] = arg_dict["SECTION"]

        if("FILTER" in arg_dict and arg_dict["FILTER"] != "NA"):
            new_arg_dict[org]["FILTER"] = arg_dict["FILTER"]
            
        sub = None
        nsub = None
        if("SUB" in arg_dict and arg_dict["SUB"] != "NA"):
            sub = arg_dict["SUB"]
            new_arg_dict[org]["SUB"] = sub
            nsub = sub


        new_arg_dict[org]["FIELDS"] = list()
        pattern_replace = "-*"
        pattern_replace1 = "*KEY"
        fields = None
        if(keys):
            for key in new_arg_dict[org]["KEY"]:
                if(nsub in key_to_fields_alias):
                    fields = key_to_fields_alias[nsub]
                else:
                    nsub = nsub.replace(key, pattern_replace1)
                    #print(nsub)
                    if(nsub in key_to_fields_alias):
                        fields = key_to_fields_alias[nsub]
                        fields = [s.replace(pattern_replace1, key.upper()) for s in fields]
                if(new_arg_dict[org]["SECTION"] == "REGULAR"):
                    fields = [s.replace(pattern_replace, "") for s in fields]
                elif(new_arg_dict[org]["SECTION"] == "REGULARFULL"):
                    fields = [s.replace(pattern_replace, "-FULLYEAR") for s in fields]
                elif(new_arg_dict[org]["SECTION"] == "GUIDE"):
                    fields = [s.replace(pattern_replace, "-GUIDE") for s in fields]
                elif(new_arg_dict[org]["SECTION"] == "GUIDEFULL"):
                    fields = [s.replace(pattern_replace, "-GUIDEFULL") for s in fields]
                break
        if(fields):
            new_arg_dict[org]["FIELDS"] = new_arg_dict[org]["FIELDS"] + fields
            
    print()
    print(new_arg_dict)
    #new_arg_dict["ALLENTS"] = allEntities
    print()
    
    return new_arg_dict

In [127]:
def getTableData(args):
    gData = dict()
    
    for org in args["ORGS"]:
        gData[org] = dict()
        gData[org]["DATA"] = dict()
        gData[org]["TEXT"] = dict()
        gData[org]["INDEX"] = list()
        #print(org)
        
        sym = args[org]["SYM"]

        allEntities = getEntities(sym)

        if not allEntities:
            return None

        attr = allEntities["ENTITIES"]
        #search = args[org]["SEARCH"]
        #if(search == "LATEST-QTR"):
        #    search = getEntAttr(attr, search)
            #print(search)
        fields = args[org]["FIELDS"]
        calendar = args[org]["CALENDAR"]
        if("COUNT" in args[org]):
            count = args[org]["COUNT"]
        else:
            count = maxEntCount

        found = list()

        gData[org]["FIELDS"] = fields
        gData[org]["CALENDAR"] = calendar
        gData[org]["SYM"] = sym
        gData[org]["RALIAS"] = args["RALIAS"]
        gData[org]["KEY"] = args[org]["KEY"]

        filter_expr = None
        nfilter_expr = None
        search = None
        if("FILTER" in args[org]):
            filter_expr = args[org]["FILTER"]
            
        fqtr = args[org]["FROM"]
        tqtr = args[org]["TO"]
        latest_qtr = getEntAttr(attr, "LATEST-QTR")
        
        if(fqtr == "ALL" and tqtr == "ALL"):
            if(calendar == "QUARTERLY"):
                search = 'Q\d+-[0-9][0-9][0-9][0-9]'
            elif(calendar == "YEARLY"):
                search = 'ALL-[0-9][0-9][0-9][0-9]'
        else:
            if(fqtr == "LATEST" or fqtr == "NEXT"):
                fqtr = latest_qtr
            if(tqtr == "LATEST" or tqtr == "NEXT"):
                tqtr = latest_qtr
            search = list_quarter_range(fqtr, tqtr, calendar)
            
        gData[org]["SEARCH"] = search
        print(gData[org]["SEARCH"])

        cnt = 0

        for item in attr:
            #print(item)
            if "PUBLISH" in attr[item] and not attr[item]["PUBLISH"]:
                continue
            if("GUIDE" not in attr[item]):
                continue
            if((isinstance(search, list) and item in search) or (not isinstance(search, list) and re.search(search, item))):
                #print(item, search)
                if(cnt > 0 and isinstance(search, list) and cnt >= len(search)):
                    break
                if(cnt > 0 and not isinstance(search, list) and ("d+" not in search and "ALL-" not in search)): # Exact search so end it here
                    break
                cnt = cnt + 1

                nattr = normalize_keys(attr[item])

                if("SECTION" in args[org]):
                    section = args[org]["SECTION"]
                    if(section == "GUIDE" or section == "GUIDEFULL"):
                        metric = "GUIDE-QTR"
                        if(metric not in gData[org]["DATA"]):
                            gData[org]["DATA"][metric] = list()
                            gData[org]["TEXT"][metric] = list()
                        guide_qtr = getEntAttr(attr[item], "GUIDE")
                        #print(guide_qtr)
                        if(guide_qtr):
                            gData[org]["DATA"][metric].append(guide_qtr)
                            gData[org]["TEXT"][metric].append(guide_qtr)
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)

                found = list()
                for metric in attr[item]:
                    #print(metric)
                    if(filter_expr):
                        nmetric = metric.replace("-","_").replace(" ","_")
                        nfilter_expr = filter_expr.replace("ANY.",nmetric+".")
                    if(metric in fields or (nfilter_expr and apply_json_filter(nattr, nfilter_expr))):
                        found.append(metric)
                        #print(item,metric)
                        if(cnt == 1):
                            if("GUIDE" not in attr[item]):
                                continue
                            if("QOQ" in metric and "GUIDE-QOQ" in fields):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                gmetric = metric.replace("-QOQ","")
                                gmetric = gmetric+"-GUIDE-QOQ"
                                if(metric not in gData[org]["DATA"]):
                                    gData[org]["DATA"][metric] = list()
                                    gData[org]["TEXT"][metric] = list()
                                #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                                #gindex = "Next Quarter"
                                #if "YEAR" in calendar:
                                #    gindex = "Next Year"
                                if gindex not in gData[org]["INDEX"]:
                                    gData[org]["INDEX"].append(gindex)
                                if gmetric in attr[item]:
                                    if("NUMBER-PCT" in attr[item][gmetric]):
                                        gData[org]["DATA"][metric].append(attr[item][gmetric]["NUMBER-PCT"])
                                        gData[org]["TEXT"][metric].append(attr[item][gmetric]["RTEXT-PCT"])
                                    else:
                                        gData[org]["DATA"][metric].append(None)
                                        gData[org]["TEXT"][metric].append("ND")
                                else:
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                            elif("YOY" in metric and "GUIDE-YOY" in fields):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                gmetric = metric.replace("-YOY","")
                                gmetric = gmetric+"-GUIDE-YOY"
                                if(metric not in gData[org]["DATA"]):
                                    gData[org]["DATA"][metric] = list()
                                    gData[org]["TEXT"][metric] = list()
                                #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                                #gindex = "Next Quarter"
                                #if "YEAR" in calendar:
                                #    gindex = "Next Year"
                                if gindex not in gData[org]["INDEX"]:
                                    gData[org]["INDEX"].append(gindex)
                                if gmetric in attr[item]:
                                    if("NUMBER-PCT" in attr[item][gmetric]):
                                        gData[org]["DATA"][metric].append(attr[item][gmetric]["NUMBER-PCT"])
                                        gData[org]["TEXT"][metric].append(attr[item][gmetric]["RTEXT-PCT"])
                                    else:
                                        gData[org]["DATA"][metric].append(None)
                                        gData[org]["TEXT"][metric].append("ND")
                                else:
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                            elif("GUIDE-CSUS" in metric and "GUIDE-CSUS-ORIG" in fields):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                gmetric = metric+"-ORIG"
                                if(metric not in gData[org]["DATA"]):
                                    gData[org]["DATA"][metric] = list()
                                    gData[org]["TEXT"][metric] = list()
                                #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                                #gindex = "Next Quarter"
                                #if "YEAR" in calendar:
                                #    gindex = "Next Year"
                                if gindex not in gData[org]["INDEX"]:
                                    gData[org]["INDEX"].append(gindex)
                                if gmetric in attr[item]:
                                    if("NUMBER-MONEY" in attr[item][gmetric]):
                                        gData[org]["DATA"][metric].append(attr[item][gmetric]["NUMBER-MONEY"])
                                        gData[org]["TEXT"][metric].append(attr[item][gmetric]["RTEXT-MONEY"])
                                    else:
                                        gData[org]["DATA"][metric].append(None)
                                        gData[org]["TEXT"][metric].append("ND")
                                else:
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                            elif("GUIDANCE" in fields):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                gmetric = metric+"-GUIDE"
                                if(metric not in gData[org]["DATA"]):
                                    gData[org]["DATA"][metric] = list()
                                    gData[org]["TEXT"][metric] = list()
                                #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                                #gindex = "Next Quarter"
                                #if "YEAR" in calendar:
                                #    gindex = "Next Year"
                                if gindex not in gData[org]["INDEX"]:
                                    gData[org]["INDEX"].append(gindex)
                                if gmetric in attr[item]:
                                    if("NUMBER-MONEY" in attr[item][gmetric]):
                                        if("NUMBER-AVG-MONEY" in attr[item][gmetric]):
                                            gData[org]["DATA"][metric].append(attr[item][gmetric]["NUMBER-AVG-MONEY"])
                                            gData[org]["TEXT"][metric].append(attr[item][gmetric]["RTEXT-AVG-MONEY"])
                                        else:
                                            gData[org]["DATA"][metric].append(attr[item][gmetric]["NUMBER-MONEY"])
                                            gData[org]["TEXT"][metric].append(attr[item][gmetric]["RTEXT-MONEY"])
                                    else:
                                        gData[org]["DATA"][metric].append(None)
                                        gData[org]["TEXT"][metric].append("ND")
                                else:
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                            if("YEARLY" == calendar and "ALL" in search):
                                continue
                        if("NUMBER-MONEY" in attr[item][metric]):
                            #print(attr[item][metric]["RTEXT-MONEY"])
                            #Metric guide for latest quarter
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)
                            if("NUMBER-AVG-MONEY" in attr[item][metric]):
                                gData[org]["DATA"][metric].append(attr[item][metric]["NUMBER-AVG-MONEY"])
                                gData[org]["TEXT"][metric].append(attr[item][metric]["RTEXT-AVG-MONEY"])
                            else:
                                gData[org]["DATA"][metric].append(attr[item][metric]["NUMBER-MONEY"])
                                gData[org]["TEXT"][metric].append(attr[item][metric]["RTEXT-MONEY"])
                            if("-RESULT" in metric):
                                if("ISBEATSTR" in attr[item][metric]):
                                    nmetric = "ISBEATSTR"
                                    gData[org]["DATA"][nmetric] = list()
                                    gData[org]["TEXT"][nmetric] = list()
                                    gData[org]["DATA"][nmetric].append(attr[item][metric]["ISBEATSTR"])
                                    gData[org]["TEXT"][nmetric].append(attr[item][metric]["ISBEATSTR"])
                                if("BEATSTR" in attr[item][metric]):
                                    nmetric = "BEATSTR"
                                    gData[org]["DATA"][nmetric] = list()
                                    gData[org]["TEXT"][nmetric] = list()
                                    gData[org]["DATA"][nmetric].append(attr[item][metric]["BEATSTR"])
                                    gData[org]["TEXT"][nmetric].append(attr[item][metric]["BEATSTR"])
                            elif(nfilter_expr and apply_json_filter(nattr, nfilter_expr)):
                                ametric = metric + "-QOQ"
                                if(ametric in attr[item] and "NUMBER-PCT" in attr[item][ametric]):
                                    gData[org]["DATA"][ametric] = list()
                                    gData[org]["TEXT"][ametric] = list()
                                    gData[org]["DATA"][ametric].append(attr[item][ametric]["NUMBER-PCT"])
                                    gData[org]["TEXT"][ametric].append(attr[item][ametric]["RTEXT-PCT"])
                                ametric = metric + "-YOY"
                                if(ametric in attr[item] and "NUMBER-PCT" in attr[item][ametric]):
                                    gData[org]["DATA"][ametric] = list()
                                    gData[org]["TEXT"][ametric] = list()
                                    gData[org]["DATA"][ametric].append(attr[item][ametric]["NUMBER-PCT"])
                                    gData[org]["TEXT"][ametric].append(attr[item][ametric]["RTEXT-PCT"])

                        elif("NUMBER-CD" in attr[item][metric]):
                            #print(attr[item][metric]["RTEXT-CD"])
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            gData[org]["DATA"][metric].append(attr[item][metric]["NUMBER-CD"])
                            gData[org]["TEXT"][metric].append(attr[item][metric]["RTEXT-CD"])
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)

                            if(nfilter_expr and apply_json_filter(nattr, nfilter_expr)):
                                ametric = metric + "-QOQ"
                                if(ametric in attr[item] and "NUMBER-PCT" in attr[item][ametric]):
                                    gData[org]["DATA"][ametric] = list()
                                    gData[org]["TEXT"][ametric] = list()
                                    gData[org]["DATA"][ametric].append(attr[item][ametric]["NUMBER-PCT"])
                                    gData[org]["TEXT"][ametric].append(attr[item][ametric]["RTEXT-PCT"])
                                ametric = metric + "-YOY"
                                if(ametric in attr[item] and "NUMBER-PCT" in attr[item][ametric]):
                                    gData[org]["DATA"][ametric] = list()
                                    gData[org]["TEXT"][ametric] = list()
                                    gData[org]["DATA"][ametric].append(attr[item][ametric]["NUMBER-PCT"])
                                    gData[org]["TEXT"][ametric].append(attr[item][ametric]["RTEXT-PCT"])

                        elif("NUMBER-PCT" in attr[item][metric]):
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            gData[org]["DATA"][metric].append(attr[item][metric]["NUMBER-PCT"])
                            if("RTEXT-PCT" in attr[item][metric]):
                                gData[org]["TEXT"][metric].append(attr[item][metric]["RTEXT-PCT"])
                            else:
                                gData[org]["TEXT"][metric].append(attr[item][metric]["TEXT-PCT"])
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)

                        elif(isinstance(attr[item][metric], list)):
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            gData[org]["DATA"][metric].append(attr[item][metric])
                            gData[org]["TEXT"][metric].append(attr[item][metric])
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)

                        else:
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            gData[org]["DATA"][metric].append(None)
                            gData[org]["TEXT"][metric].append("ND")
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)
                if(len(found)>0 and (len(found) < len(fields))):
                    nf = set(found) ^ set(fields)
                    nf = list(nf)
                    #print(found, fields, nf)
                    for metric in nf:
                        if(metric != "GUIDANCE" and metric != "GUIDE-YOY" and metric != "GUIDE-QOQ" and metric != "GUIDE-CSUS-ORIG"):
                            if(cnt == 1):
                                if("GUIDE-YOY" in nf and "YOY" in metric):
                                    # Increase count to make room for guidance
                                    count = count + 1
                                    gindex = attr[item]["GUIDE"]+"-GUIDE"
                                    if(metric not in gData[org]["DATA"]):
                                        gData[org]["DATA"][metric] = list()
                                        gData[org]["TEXT"][metric] = list()
                                    if gindex not in gData[org]["INDEX"]:
                                        gData[org]["INDEX"].append(gindex)
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                                elif("GUIDE-QOQ" in nf and "QOQ" in metric):
                                    # Increase count to make room for guidance
                                    count = count + 1
                                    gindex = attr[item]["GUIDE"]+"-GUIDE"
                                    if(metric not in gData[org]["DATA"]):
                                        gData[org]["DATA"][metric] = list()
                                        gData[org]["TEXT"][metric] = list()
                                    if gindex not in gData[org]["INDEX"]:
                                        gData[org]["INDEX"].append(gindex)
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                                elif("GUIDANCE" in nf):
                                    # Increase count to make room for guidance
                                    count = count + 1
                                    gindex = attr[item]["GUIDE"]+"-GUIDE"
                                    if(metric not in gData[org]["DATA"]):
                                        gData[org]["DATA"][metric] = list()
                                        gData[org]["TEXT"][metric] = list()
                                    if gindex not in gData[org]["INDEX"]:
                                        gData[org]["INDEX"].append(gindex)
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")

                                if(calendar == "YEARLY"):
                                    #print("CONTINUING...", metric)
                                    if(metric not in gData[org]["DATA"]):
                                        gData[org]["DATA"][metric] = list()
                                        gData[org]["TEXT"][metric] = list()
                                    gData[org]["DATA"][metric].append(None)
                                    gData[org]["TEXT"][metric].append("ND")
                                    continue
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            gData[org]["DATA"][metric].append(None)
                            gData[org]["TEXT"][metric].append("ND")

                if(fields[0] in gData[org]["DATA"] and len(gData[org]["DATA"][fields[0]]) >= count):
                    break

                if(len(found) == 0):
                    for metric in fields:
                        if(metric != "GUIDANCE" and metric != "GUIDE-YOY" and metric != "GUIDE-QOQ" and metric != "GUIDE-CSUS-ORIG"):
                            if(metric not in gData[org]["DATA"]):
                                gData[org]["DATA"][metric] = list()
                                gData[org]["TEXT"][metric] = list()
                            gData[org]["DATA"][metric].append(None)
                            gData[org]["TEXT"][metric].append("ND")
                            nitem = item
                            nitem = item.replace("ALL-","")
                            if nitem not in gData[org]["INDEX"]:
                                gData[org]["INDEX"].append(nitem)

        for metric in fields:
            if metric in gData[org]["DATA"] and metric in gData[org]["TEXT"]:
                gData[org]["DATA"][metric].reverse()
                gData[org]["TEXT"][metric].reverse()

        if "INDEX" in gData[org]:
            gData[org]["INDEX"].reverse()
            
    #print(gData)
    return gData

In [128]:
def showTable(gData):
    if not gData:
        return None
    #print(gData)
    for org in gData:
        if "DATA" not in gData[org]:
            return None
        if not gData[org]["DATA"]:
            print("Empty Data")
            return None
        if "INDEX" not in gData[org]:
            return None
        df = pd.DataFrame(data=gData[org]["TEXT"], index=gData[org]["INDEX"])
        df = df.replace("ND", np.nan)
        df = df.dropna(axis=1, how='all')
        df = df.dropna(axis=0, how='all')
        df = df.replace(np.nan, "NA")
        #print(df)
        print("\nSee table below for {}".format(org))
        display(df.T)

In [129]:
def showtxt(gData, tnum):
    for org in gData:
        ansAttr = dict()
        ansAttr["*ORG"] = org.title()
        ralias = gData[org]["RALIAS"]
        #print(gData[org]["TEXT"])
        ans = None
        ntnum = int(tnum)
        if(ntnum >= 3):
            ans = ansTemplateStr[str(tnum)]
        for index, item in enumerate(gData[org]["INDEX"]):
            qtr = item.split("-")[0]
            ansAttr["*QTR"] = qtr
            yr = item.split("-")[1]
            ansAttr["*YEAR"] = yr
            #print(qtr, yr)
            if(ntnum == 2):
                ans_text = None
                ans_cnt = 0
            for metric in gData[org]["TEXT"]:
                #print(metric)
                if(isinstance(gData[org]["TEXT"][metric][index], list) and (ntnum) == 2):
                    origMetricName = metric
                    if(metric in ralias):
                        origMetricName = ralias[metric]
                    elif(metric in key_alias):
                        origMetricName = key_alias[metric]
                    ansAttr["*METRIC"] = origMetricName.title()
                    ansAttr["*COUNT"] = (len(gData[org]["TEXT"][metric][index]))
                    liststr = "\n".join(gData[org]["TEXT"][metric][index])
                    ansAttr["*RESULT"] = liststr
                    #print("{} of {} in quarter {} year {} are:\n\n{}".format(origMetricName, org, qtr, yr, liststr))
                    if not ans:
                        ans = ansTemplateStr[str(tnum)]
                    else:
                        ans = ans + ansTemplateStr[str(tnum)]
                    for attr in ansAttr:
                        if(attr in ans):
                            ans = ans.replace(attr, str(ansAttr[attr]))
                else:
                    if(ntnum == 2):
                        if(gData[org]["TEXT"][metric][index] != "ND"):
                            val = gData[org]["TEXT"][metric][index]
                            if("YOY" in metric):
                                if(val.startswith("(")):
                                    chg = "DECLINED"
                                else:
                                    chg = "GREW"
                                nmetric = metric.replace("-YOY","")
                                origMetricName = nmetric
                                if(nmetric in ralias):
                                    origMetricName = ralias[nmetric]
                                origMetricName = origMetricName.title()
                                if ans_text:
                                    ans_text = ans_text + " It " + chg + " " + val + " year over year."
                            elif("QOQ" in metric):
                                if(val.startswith("(")):
                                    chg = "DECLINED"
                                else:
                                    chg = "GREW"
                                nmetric = metric.replace("-QOQ","")
                                origMetricName = nmetric
                                if(nmetric in ralias):
                                    origMetricName = ralias[nmetric]
                                origMetricName = origMetricName.title()
                                if ans_text:
                                    ans_text = ans_text + " It " + chg + " " + val + " quarter over quarter."
                            else:
                                ans_cnt = ans_cnt + 1
                                origMetricName = metric
                                if(metric in ralias):
                                    origMetricName = ralias[metric]
                                origMetricName = origMetricName.title()
                                if not ans_text:
                                    ans_text = origMetricName + " is " + val + " in " + qtr + " " + yr + "."
                                else:
                                    ans_text = ans_text + "\n" + origMetricName + " is " + val + " in " + qtr + " " + yr + "."
                    elif((ntnum) >= 3):
                        if(gData[org]["TEXT"][metric][index] != "ND"):
                            if("-RESULT" in metric):
                                ansAttr["*MARGIN"] = gData[org]["TEXT"][metric][index]
                                ansAttr["*METRIC"] = metric.replace("-RESULT", "").replace("-GUIDE", "").replace("FULL","")
                            elif(metric == "BEATSTR"):
                                ansAttr["*CHG"] = gData[org]["TEXT"][metric][index]
                            elif(metric == "ISBEATSTR"):
                                ansAttr["*RESULT"] = gData[org]["TEXT"][metric][index]
                            elif(metric == "GUIDE-QTR"):
                                guide_qtr = gData[org]["TEXT"][metric][index]
                                gqtr = guide_qtr.split("-")[0]
                                ansAttr["*GQTR"] = gqtr
                                gyr = guide_qtr.split("-")[1]
                                ansAttr["*GYR"] = gyr
                            else:
                                ansAttr["*NUMBER"] = gData[org]["TEXT"][metric][index]
                    else:
                        if(gData[org]["TEXT"][metric][index] != "ND"):
                            val = gData[org]["TEXT"][metric][index]
                            if(val.startswith("(")):
                                chg = "declined"
                            else:
                                chg = "grew"
                            if("YOY" in metric):
                                nmetric = metric.replace("-YOY","")
                                origMetricName = nmetric
                                if(nmetric in ralias):
                                    origMetricName = ralias[nmetric]
                                origMetricName = origMetricName.title()
                                ans = ans + origMetricName + " " + chg + " " + val + " year over year. "
                            elif("QOQ" in metric):
                                nmetric = metric.replace("-QOQ","")
                                origMetricName = nmetric
                                if(nmetric in ralias):
                                    origMetricName = ralias[nmetric]
                                origMetricName = origMetricName.title()
                                ans = ans + origMetricName + " " + chg + " " + val + " quarter over quarter. "
                            else:
                                origMetricName = metric
                                if(metric in ralias):
                                    origMetricName = ralias[metric]
                                ansAttr["*METRIC"] = origMetricName.title()
                                if not ans:
                                    ans = ansTemplateStr[str(tnum)]
                                else:
                                    ans = ans + ansTemplateStr[str(tnum)]
                                ansAttr["*RESULT"] = val
                                for attr in ansAttr:
                                    if(attr in ans):
                                        ans = ans.replace(attr, str(ansAttr[attr]))
        if(ans or ans_text):
            if((ntnum) >=3):
                for attr in ansAttr:
                    if(attr in ans):
                        ans = ans.replace(attr, str(ansAttr[attr]))
            elif(ntnum == 2 and ans_text):
                #print(ans_text)
                ansAttr["*METRIC"] = gData[org]["KEY"][0]
                ansAttr["*COUNT"] = ans_cnt
                ansAttr["*RESULT"] = ans_text
                ans = ansTemplateStr[str(tnum)]
                for attr in ansAttr:
                    if(attr in ans):
                        ans = ans.replace(attr, str(ansAttr[attr]))
                
    #if ans and ("*" not in ans):
    if ans:
        return ans
    else:
        return naStr

In [130]:
def getAns(genData, gData):
    ans = genData["ANS"]
    asplit = ans.split("!!")
    away = asplit[0].split(":")[1]
    tnum = asplit[1].split(":")[1]
    if(away == "SHOWTXT"):
        return showtxt(gData, tnum)
    elif(away == "SHOWTBL"):
        showTable(gData)
        return None

In [131]:
qa = [
        "List all positive facts of Appian from Quarter Q2 Year 2025.",
        "List all negative facts of Appian from Quarter Q4 Year 2024.",
        "Did Appian beat revenue expectation in latest quarter?",
        "Did Appian beat EPS expectation in latest quarter?",
        "Did Appian beat EPS guidance for next quarter in reported quarter?",
        "Did Appain beat Revenue guidance for next quarter?",
        "Did Appian raise revenue guidance for full fiscal year in latest reported quarter?",
        "Did Appian raise EPS guidance for full fiscal year in latest reported quarter?",
        "List all operational metrics of Appian from latest reported quarter.",
        "Show quarterly revenue growth table of UIPath.",
        "Show quarterly gaap-eps growth table of UIPath.",
        "Show yearly revenue growth table of UIPath.",
        "Compare quarterly revenue growth between Palantir and UIPath from Q4 2022.",
        "Show progress of yearly revenue vs yearly guidance of Appian in each quarter in table format.",
        "What was the revenue of Nvidia for Q2 2026?",
        "What was the YoY EPS growth of Cloudflare in quarter Q1 2024?",
        "List all companies those beat revenue estimates in current quarter in table format.",
        "Show all operational metrics Qoq trend of Gitlab in table format.",
        "What was the last reported quarter of Zoom?"
        #"Show all positive and negative facts count of Trade Desk for last 3 years in chart"
]
print(qa)

qargs = dict()
qargs["ARGS"] = list()
qargs["INTENT"] = list()
qargs["ANS"] = list()
qargs["ARGS"].append("SUB:POSITIVE FACTS!!KEY:POSITIVE FACTS!!ORG:APPIAN!!FROM:Q2-2025!!TO:Q2-2025!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:2")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:NEGATIVE FACTS!!KEY:NEGATIVE FACTS!!ORG:APPIAN!!FROM:Q4-2024!!TO:Q4-2024!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:2")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:BEAT REVENUE EXPECTATION?!!KEY:REVENUE!!ORG:APPIAN!!FROM:LATEST!!TO:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:3")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:BEAT EPS EXPECTATION?!!KEY:EPS!!ORG:APPIAN!!FROM:LATEST!!TO:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:3")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:BEAT EPS GUIDANCE?!!KEY:EPS!!ORG:APPIAN!!FROM:NEXT!!TO:NEXT!!CALENDAR:Q!!FILTER:NA!!SECTION:GUIDE!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:4")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:BEAT REVENUE GUIDANCE?!!KEY:REVENUE!!ORG:APPIAN!!FROM:NEXT!!TO:NEXT!!CALENDAR:Q!!FILTER:NA!!SECTION:GUIDE!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:4")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:RAISE REVENUE GUIDANCE?!!KEY:REVENUE!!ORG:APPIAN!!FROM:LATEST!!TO:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:GUIDEFULL!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:5")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:RAISE EPS GUIDANCE?!!KEY:EPS!!ORG:APPIAN!!FROM:LATEST!!TO:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:GUIDEFULL!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:5")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append('SUB:OPERATIONAL METRICS!!KEY:OPERATIONAL METRICS!!ORG:APPIAN!!FROM:LATEST!!TO:LATEST!!CALENDAR:Q!!FILTER:ANY.TEXT_METRICTYPE=="OPMETRIC"!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY')
qargs["ANS"].append("FN:SHOWTXT!!T:2")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:REVENUE GROWTH!!KEY:REVENUE!!ORG:UIPATH!!FROM:ALL!!TO:ALL!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTBL!!T:0")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:GAAP-EPS GROWTH!!KEY:GAAP-EPS!!ORG:UIPATH!!FROM:ALL!!TO:ALL!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTBL!!T:0")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:REVENUE GROWTH YEARLY!!KEY:REVENUE!!ORG:UIPATH!!FROM:ALL!!TO:ALL!!CALENDAR:Y!!FILTER:NA!!SECTION:REGULARFULL!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTBL!!T:0")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:REVENUE GROWTH!!KEY:REVENUE!!ORG:PALANTIR&&UIPATH!!FROM:Q4-2022!!TO:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTBL!!T:0")
qargs["INTENT"].append("COMPARE")

qargs["ARGS"].append("SUB:YEARLY REVENUE VS YEARLY GUIDANCE!!KEY:REVENUE!!ORG:APPIAN!!FROM:ALL!!TO:ALL!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTBL!!T:0")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:REVENUE!!KEY:REVENUE!!ORG:NVIDIA!!FROM:Q2-2026!!TO:Q2-2026!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:1")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("SUB:YOY EPS GROWTH!!KEY:EPS!!ORG:CLOUDFLARE!!FROM:Q1-2024!!TO:Q1-2024!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:1")
qargs["INTENT"].append("INFO")

qargs["ARGS"].append("KEY:REVENUE-RESULT!!ORG:NA!!QTR:CURRENT!!YEAR:CURRENT!!CALENDAR:Q!!FILTER:VAL==TRUE!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("COLS:LATEST QTR,BEAT?!!FN:SHOWTBL")
qargs["INTENT"].append("GROUP INFO")
qargs["ARGS"].append("KEY:NA!!ORG:GITLAB!!QTR:NA!!YEAR:NA!!CALENDAR:QOQ!!FILTER:KEY==OPERATIONAL!!SECTION:REGULAR!!HOW:FILTER!!SOURCE:ENTITY")
qargs["ANS"].append("COLS:QTR!!FN:SHOWTBL")
qargs["INTENT"].append("INFO")
qargs["ARGS"].append("KEY:NA!!ORG:ZOOM!!QTR:LATEST!!YEAR:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("TXT:Last reported quarter of *ORG was *QTR *YEAR.")
qargs["INTENT"].append("INFO")
#qargs["ARGS"].append("KEY:POSTIVE FACTS AND NEGATIVE FACTS!!ORG:Trade Desk!!QTR:NA!!YEAR:LAST 3!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:COUNT!!SOURCE:ENTITY")
#qargs["ANS"].append("FN:SHOWCHT")
#qargs["INTENT"].append("INFO")
print(json.dumps(qargs))

['List all positive facts of Appian from Quarter Q2 Year 2025.', 'List all negative facts of Appian from Quarter Q4 Year 2024.', 'Did Appian beat revenue expectation in latest quarter?', 'Did Appian beat EPS expectation in latest quarter?', 'Did Appian beat EPS guidance for next quarter in reported quarter?', 'Did Appain beat Revenue guidance for next quarter?', 'Did Appian raise revenue guidance for full fiscal year in latest reported quarter?', 'Did Appian raise EPS guidance for full fiscal year in latest reported quarter?', 'List all operational metrics of Appian from latest reported quarter.', 'Show quarterly revenue growth table of UIPath.', 'Show quarterly gaap-eps growth table of UIPath.', 'Show yearly revenue growth table of UIPath.', 'Compare quarterly revenue growth between Palantir and UIPath from Q4 2022.', 'Show progress of yearly revenue vs yearly guidance of Appian in each quarter in table format.', 'What was the revenue of Nvidia for Q2 2026?', 'What was the YoY EPS gro

In [133]:
qcnt = 0
for q,intent,args,ans in zip(qa, qargs["INTENT"],qargs["ARGS"],qargs["ANS"]):
    #print(q)
    #print(intent) 
    #print(args) 
    #print(ans)
    #print()
    
    qcnt = qcnt + 1
    
    genData = dict()
    genData["INTENT"] = intent
    genData["ARGS"] = args
    genData["QUESTION"] = q
    genData["ANS"] = ans
    print(genData)
    print()
    
    new_args = buildArgsFromGenData(genData)
    #print(org_to_sym)
    if not new_args:
        print(">>>",q)
        print()
        print(naStr)
        if(qcnt >= 2):
            break
        continue
    
    gData = getTableData(new_args)
    #print(gData)
    if gData is None:
        print(">>>",q)
        print()
        print(naStr)
        if(qcnt >= 2):
            break
        continue

    print()
    print(">>>",q)
    #print()
    answer = getAns(genData, gData)
    if(answer):
        print(answer)
        print()
    
    if(qcnt >= 14):
        break

{'INTENT': 'INFO', 'ARGS': 'SUB:POSITIVE FACTS!!KEY:POSITIVE FACTS!!ORG:APPIAN!!FROM:Q2-2025!!TO:Q2-2025!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY', 'QUESTION': 'List all positive facts of Appian from Quarter Q2 Year 2025.', 'ANS': 'FN:SHOWTXT!!T:2'}

{'SUB': 'POSITIVE FACTS', 'KEY': 'POSITIVE FACTS', 'ORG': 'APPIAN', 'FROM': 'Q2-2025', 'TO': 'Q2-2025', 'CALENDAR': 'Q', 'FILTER': 'NA', 'SECTION': 'REGULAR', 'HOW': 'EXACT', 'SOURCE': 'ENTITY'}

{'ORGS': ['APPIAN'], 'APPIAN': {'SYM': 'APPN', 'KEY': ['POSITIVE FACTS'], 'CALENDAR': 'QUARTERLY', 'FROM': 'Q2-2025', 'TO': 'Q2-2025', 'SECTION': 'REGULAR', 'SUB': 'POSITIVE FACTS', 'FIELDS': ['POSFACTS']}, 'RALIAS': {'POSITIVE FACTS': 'POSITIVE FACTS'}}

['Q2-2025']

>>> List all positive facts of Appian from Quarter Q2 Year 2025.
Appian reported 38 Positive Facts in quarter Q2 year 2025, these were:
GAAP-EPS GREW 105.26% QUARTER OVER QUARTER IN Q2 2025
GAAP-EPS GREW 100.17% YEAR OVER YEAR IN Q2 2025
GAAP GROSS PROFIT GRE

['Q2-2025']

>>> List all operational metrics of Appian from latest reported quarter.
Appian reported 4 OPERATIONAL METRICS in quarter Q2 year 2025, these were:
Subscription Revenue-Cloud is $106.9MN in Q2 2025. It GREW 7.11% quarter over quarter. It GREW 20.93% year over year.
Subscriptions Revenue is $132.7MN in Q2 2025. It DECLINED (1.26%) quarter over quarter. It GREW 17.43% year over year.
Professional Services Revenue is $38.0MN in Q2 2025. It GREW 18.38% quarter over quarter. It GREW 13.43% year over year.
Net Dollar Expansion is 111.0% in Q2 2025.

{'INTENT': 'INFO', 'ARGS': 'SUB:REVENUE GROWTH!!KEY:REVENUE!!ORG:UIPATH!!FROM:ALL!!TO:ALL!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY', 'QUESTION': 'Show quarterly revenue growth table of UIPath.', 'ANS': 'FN:SHOWTBL!!T:0'}

{'SUB': 'REVENUE GROWTH', 'KEY': 'REVENUE', 'ORG': 'UIPATH', 'FROM': 'ALL', 'TO': 'ALL', 'CALENDAR': 'Q', 'FILTER': 'NA', 'SECTION': 'REGULAR', 'HOW': 'EXACT', 'SOURCE': 'ENTITY'}

{'ORGS': 

,Q1-2023,Q2-2023,Q3-2023,Q4-2023,Q1-2024,Q2-2024,Q3-2024,Q4-2024,Q1-2025,Q2-2025,Q3-2025,Q4-2025,Q1-2026,Q2-2026,Q3-2026-GUIDE
REVENUE,$245.1MN,$242.2MN,$262.7MN,$308.5MN,$289.6MN,$287.3MN,$326MN,$405MN,$335MN,$316MN,$355MN,$424MN,$357MN,$362MN,$392MN
REVENUE-QOQ,(15.4%),(1.18%),8.46%,17.43%,(6.13%),(0.79%),13.47%,24.23%,(17.28%),(5.67%),12.34%,19.44%,(15.8%),1.4%,8.43%
REVENUE-YOY,31.62%,23.87%,18.98%,6.49%,18.32%,18.5%,24.1%,31.28%,15.68%,9.99%,8.9%,4.69%,6.57%,14.56%,10.56%


{'INTENT': 'INFO', 'ARGS': 'SUB:GAAP-EPS GROWTH!!KEY:GAAP-EPS!!ORG:UIPATH!!FROM:ALL!!TO:ALL!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY', 'QUESTION': 'Show quarterly gaap-eps growth table of UIPath.', 'ANS': 'FN:SHOWTBL!!T:0'}

{'SUB': 'GAAP-EPS GROWTH', 'KEY': 'GAAP-EPS', 'ORG': 'UIPATH', 'FROM': 'ALL', 'TO': 'ALL', 'CALENDAR': 'Q', 'FILTER': 'NA', 'SECTION': 'REGULAR', 'HOW': 'EXACT', 'SOURCE': 'ENTITY'}

{'ORGS': ['UIPATH'], 'UIPATH': {'SYM': 'PATH', 'KEY': ['GAAP-EPS'], 'CALENDAR': 'QUARTERLY', 'FROM': 'ALL', 'TO': 'ALL', 'SECTION': 'REGULAR', 'SUB': 'GAAP-EPS GROWTH', 'FIELDS': ['GAAP-EPS', 'GAAP-EPS-QOQ', 'GAAP-EPS-YOY', 'GUIDANCE', 'GUIDE-QOQ', 'GUIDE-YOY']}, 'RALIAS': {'GAAP-EPS': 'GAAP-EPS'}}

Q\d+-[0-9][0-9][0-9][0-9]

>>> Show quarterly gaap-eps growth table of UIPath.

See table below for UIPATH


,Q1-2023,Q2-2023,Q3-2023,Q4-2023,Q1-2024,Q2-2024,Q3-2024,Q4-2024,Q1-2025,Q2-2025,Q3-2025,Q4-2025,Q1-2026,Q2-2026
GAAP-EPS,-$0.23,-$0.22,-$0.1,-$0.05,-$0.06,-$0.11,-$0.06,$0.06,-$0.05,-$0.15,-$0.02,$0.09,-$0.04,$0.0
GAAP-EPS-QOQ,NA,4.37%,54.79%,50.51%,(20.41%),(84.75%),45.87%,203.39%,(180.33%),(204.08%),87.25%,578.95%,(142.86%),102.56%
GAAP-EPS-YOY,79.35%,(15.87%),NA,NA,74.24%,50.23%,40.4%,224.49%,16.95%,(36.7%),67.8%,49.18%,20.41%,100.67%


...

,Q4-2022,Q1-2023,Q2-2023,Q3-2023,Q4-2023,Q1-2024,Q2-2024,Q3-2024,Q4-2024,Q1-2025,Q2-2025,Q3-2025-GUIDE
REVENUE,$509MN,$525MN,$533MN,$558MN,$608MN,$634MN,$678MN,$726MN,$828MN,$884MN,$1BN,$1.08BN
REVENUE-QOQ,6.43%,3.26%,1.55%,4.63%,8.96%,4.33%,6.9%,6.99%,14.06%,6.81%,13.56%,8.1%
REVENUE-YOY,17.59%,17.71%,12.68%,16.77%,19.54%,20.72%,27.13%,30.11%,36.18%,39.36%,48.05%,49.55%



See table below for UIPATH


,Q1-2023,Q2-2023,Q3-2023,Q4-2023,Q1-2024,Q2-2024,Q3-2024,Q4-2024,Q1-2025,Q2-2025,Q3-2025,Q4-2025,Q1-2026,Q2-2026,Q3-2026-GUIDE
REVENUE,$245.1MN,$242.2MN,$262.7MN,$308.5MN,$289.6MN,$287.3MN,$326MN,$405MN,$335MN,$316MN,$355MN,$424MN,$357MN,$362MN,$392MN
REVENUE-QOQ,(15.4%),(1.18%),8.46%,17.43%,(6.13%),(0.79%),13.47%,24.23%,(17.28%),(5.67%),12.34%,19.44%,(15.8%),1.4%,8.43%
REVENUE-YOY,31.62%,23.87%,18.98%,6.49%,18.32%,18.5%,24.1%,31.28%,15.68%,9.99%,8.9%,4.69%,6.57%,14.56%,10.56%


{'INTENT': 'INFO', 'ARGS': 'SUB:YEARLY REVENUE VS YEARLY GUIDANCE!!KEY:REVENUE!!ORG:APPIAN!!FROM:ALL!!TO:ALL!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY', 'QUESTION': 'Show progress of yearly revenue vs yearly guidance of Appian in each quarter in table format.', 'ANS': 'FN:SHOWTBL!!T:0'}

{'SUB': 'YEARLY REVENUE VS YEARLY GUIDANCE', 'KEY': 'REVENUE', 'ORG': 'APPIAN', 'FROM': 'ALL', 'TO': 'ALL', 'CALENDAR': 'Q', 'FILTER': 'NA', 'SECTION': 'REGULAR', 'HOW': 'EXACT', 'SOURCE': 'ENTITY'}

{'ORGS': ['APPIAN'], 'APPIAN': {'SYM': 'APPN', 'KEY': ['REVENUE'], 'CALENDAR': 'QUARTERLY', 'FROM': 'ALL', 'TO': 'ALL', 'SECTION': 'REGULAR', 'SUB': 'YEARLY REVENUE VS YEARLY GUIDANCE', 'FIELDS': ['REVENUE-FULLYEAR', 'REVENUE-FULLYEAR-YOY', 'REVENUE-GUIDEFULL', 'REVENUE-GUIDEFULL-YOY']}, 'RALIAS': {'REVENUE': 'REVENUE'}}

Q\d+-[0-9][0-9][0-9][0-9]

>>> Show progress of yearly revenue vs yearly guidance of Appian in each quarter in table format.

See table below for APPIAN


,Q3-2022,Q4-2022,Q1-2023,Q2-2023,Q3-2023,Q4-2023,Q1-2024,Q2-2024,Q3-2024,Q4-2024,Q1-2025,Q2-2025
REVENUE-GUIDEFULL,$464MN,$532MN,$536MN,$540MN,$540MN,$616MN,$616MN,$612MN,$614MN,$682MN,$684MN,$699MN
REVENUE-GUIDEFULL-YOY,25.51%,13.78%,14.42%,15.49%,15.49%,12.94%,12.94%,12.3%,12.58%,10.53%,10.86%,13.29%
REVENUE-FULLYEAR-YOY,NA,26.73%,NA,NA,NA,16.54%,NA,NA,NA,13.13%,NA,NA
REVENUE-FULLYEAR,NA,$468.0MN,NA,NA,NA,$545.4MN,NA,NA,NA,$617.0MN,NA,NA
